# RAG Math and Retrieval — Theory Notebook

[← Quantization and Distillation](../11-Quantization-and-Distillation/notes.md) | [Home](../../README.md) | [Serving and Systems Tradeoffs →](../13-Serving-and-Systems-Tradeoffs/notes.md)

Interactive implementations of retrieval-augmented generation mathematics.

**Sections:**
1. TF-IDF and BM25 Sparse Retrieval
2. Dense Retrieval and Contrastive Loss
3. Similarity Functions Compared
4. Inverted File Index (IVF) Simulation
5. Product Quantization (PQ) for Vector Compression
6. HNSW Graph Search Simulation
7. HyDE — Hypothetical Document Embeddings
8. Reciprocal Rank Fusion (RRF)
9. NDCG, MRR, and Retrieval Metrics
10. RAG Marginalisation and Probabilistic Retrieval

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_width=14):
    """Print a formatted ASCII table."""
    hdr = ' | '.join(h.center(col_width) for h in headers)
    print(hdr)
    print('-' * len(hdr))
    for row in rows:
        print(' | '.join(str(v).center(col_width) for v in row))

def fmt_sci(x, p=4):
    return f'{x:.{p}e}'

def fmt_vec(v, p=4):
    return '[' + ', '.join(f'{x:.{p}f}' for x in v) + ']'

def fmt_bytes(n_bytes):
    if n_bytes >= 1e9:
        return f'{n_bytes/1e9:.2f} GB'
    elif n_bytes >= 1e6:
        return f'{n_bytes/1e6:.2f} MB'
    elif n_bytes >= 1e3:
        return f'{n_bytes/1e3:.2f} KB'
    return f'{n_bytes} B'

def softmax(x, temperature=1.0):
    """Numerically stable softmax with temperature."""
    x = np.asarray(x, dtype=np.float64)
    z = x / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def kl_divergence(p, q):
    """KL(P || Q) with numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    mask = p > 1e-15
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def entropy(p):
    """Shannon entropy in nats."""
    p = np.asarray(p, dtype=np.float64)
    mask = p > 1e-15
    return -np.sum(p[mask] * np.log(p[mask]))

print('Setup complete ✓')

## §1 TF-IDF and BM25 Sparse Retrieval

Classic lexical retrieval from first principles.

**TF-IDF weight:**
$$w(t,d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}} \cdot \log\frac{|\mathcal{D}|}{|\{d : t \in d\}| + 1}$$

**BM25 score:**
$$\text{BM25}(q,d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f_{t,d}(k_1+1)}{f_{t,d} + k_1(1-b+b\cdot|d|/\text{avgdl})}$$

We implement both from scratch and compare.

In [ ]:
# === §1: TF-IDF and BM25 Sparse Retrieval ===

# --- 1a: Build a small corpus ---
corpus = [
    "neural networks learn representations from data using backpropagation",
    "information retrieval finds relevant documents given a query",
    "neural retrieval models use dense embeddings for semantic search",
    "BM25 is a classical retrieval algorithm based on term frequency",
    "transformers enable neural language models with attention mechanisms",
    "dense retrieval outperforms BM25 on semantic similarity tasks",
]
query = "neural retrieval"

print("Corpus:")
for i, doc in enumerate(corpus):
    print(f"  D{i}: {doc}")
print(f"\nQuery: '{query}'")

# --- 1b: Tokenisation (simple whitespace) ---
def tokenize(text):
    return text.lower().split()

corpus_tokens = [tokenize(d) for d in corpus]
query_tokens = tokenize(query)
vocab = sorted(set(t for doc in corpus_tokens for t in doc))
print(f"\nVocabulary size: {len(vocab)}")

# --- 1c: TF-IDF implementation ---
def compute_tf(tokens):
    """Term frequency: count / total tokens."""
    tf = {}
    for t in tokens:
        tf[t] = tf.get(t, 0) + 1
    total = len(tokens)
    return {t: c / total for t, c in tf.items()}

def compute_idf(corpus_tokens, vocab):
    """Inverse document frequency: log(N / (df + 1))."""
    N = len(corpus_tokens)
    idf = {}
    for t in vocab:
        df = sum(1 for doc in corpus_tokens if t in doc)
        idf[t] = np.log(N / (df + 1))
    return idf

idf = compute_idf(corpus_tokens, vocab)
print("\nIDF for query terms:")
for t in query_tokens:
    print(f"  IDF('{t}') = {idf.get(t, 0):.4f}")

# --- 1d: TF-IDF scoring ---
def tfidf_score(query_tokens, doc_tokens, idf):
    tf = compute_tf(doc_tokens)
    score = 0.0
    for t in query_tokens:
        if t in tf:
            score += tf[t] * idf.get(t, 0)
    return score

print("\nTF-IDF scores:")
tfidf_scores = []
for i, doc_tokens in enumerate(corpus_tokens):
    s = tfidf_score(query_tokens, doc_tokens, idf)
    tfidf_scores.append(s)
    print(f"  D{i}: {s:.4f}")

# --- 1e: BM25 implementation ---
def bm25_score(query_tokens, doc_tokens, corpus_tokens, k1=1.5, b=0.75):
    """BM25 scoring with standard parameters."""
    N = len(corpus_tokens)
    avgdl = np.mean([len(d) for d in corpus_tokens])
    dl = len(doc_tokens)
    
    doc_tf = {}
    for t in doc_tokens:
        doc_tf[t] = doc_tf.get(t, 0) + 1
    
    score = 0.0
    for t in query_tokens:
        if t not in doc_tf:
            continue
        f = doc_tf[t]
        df = sum(1 for d in corpus_tokens if t in d)
        # BM25 IDF variant
        idf_t = np.log((N - df + 0.5) / (df + 0.5))
        # BM25 TF saturation
        tf_component = (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))
        score += idf_t * tf_component
    return score

print("\nBM25 scores (k1=1.5, b=0.75):")
bm25_scores = []
for i, doc_tokens in enumerate(corpus_tokens):
    s = bm25_score(query_tokens, doc_tokens, corpus_tokens)
    bm25_scores.append(s)
    print(f"  D{i}: {s:.4f}")

# --- 1f: Compare rankings ---
tfidf_ranking = np.argsort(tfidf_scores)[::-1]
bm25_ranking = np.argsort(bm25_scores)[::-1]

print("\nRanking comparison:")
headers = ['Rank', 'TF-IDF', 'Score', 'BM25', 'Score']
rows = []
for rank in range(len(corpus)):
    ti = tfidf_ranking[rank]
    bi = bm25_ranking[rank]
    rows.append([rank+1, f'D{ti}', f'{tfidf_scores[ti]:.4f}',
                 f'D{bi}', f'{bm25_scores[bi]:.4f}'])
print_table(headers, rows, col_width=12)

# --- 1g: BM25 parameter sensitivity ---
print("\n--- BM25 Parameter Sensitivity ---")
print("Effect of k1 (TF saturation) on D2 score:")
for k1 in [0.5, 1.0, 1.5, 2.0, 3.0]:
    s = bm25_score(query_tokens, corpus_tokens[2], corpus_tokens, k1=k1, b=0.75)
    print(f"  k1={k1:.1f}: {s:.4f}")

print("\nEffect of b (length norm) on D2 score:")
for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
    s = bm25_score(query_tokens, corpus_tokens[2], corpus_tokens, k1=1.5, b=b)
    print(f"  b={b:.2f}: {s:.4f}")

## §2 Dense Retrieval and Contrastive Loss

Bi-encoder architecture with InfoNCE contrastive loss:

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B} \log \frac{\exp(\text{sim}(q_i, p_i^+)/\tau)}{\sum_{j=1}^{B} \exp(\text{sim}(q_i, p_j)/\tau)}$$

We simulate query and document encoders and train with contrastive loss.

In [ ]:
# === §2: Dense Retrieval and Contrastive Loss ===

# --- 2a: Simulate bi-encoder embeddings ---
d = 64  # embedding dimension
B = 8   # batch size

# Generate query and positive passage embeddings
# Similar pairs should have high cosine similarity
np.random.seed(42)
base_vecs = np.random.randn(B, d)

# Queries: slight perturbation of base
queries = base_vecs + 0.1 * np.random.randn(B, d)
# Positive passages: different perturbation of same base
positives = base_vecs + 0.1 * np.random.randn(B, d)

# L2-normalise
def l2_normalise(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / (norms + 1e-10)

queries = l2_normalise(queries)
positives = l2_normalise(positives)

# --- 2b: Similarity matrix ---
sim_matrix = queries @ positives.T  # [B, B] cosine similarities
print("Similarity matrix (query_i × passage_j):")
print("         ", '  '.join(f'P{j}' for j in range(B)))
for i in range(B):
    row = '  '.join(f'{sim_matrix[i,j]:+.3f}' for j in range(B))
    print(f"  Q{i}:  {row}")

print(f"\nDiagonal (positive pairs) mean: {np.mean(np.diag(sim_matrix)):.4f}")
print(f"Off-diagonal (negatives) mean:  {np.mean(sim_matrix - np.diag(np.diag(sim_matrix))):.4f}")

# --- 2c: InfoNCE loss computation ---
def infonce_loss(sim_matrix, temperature=0.1):
    """Compute InfoNCE contrastive loss."""
    B = sim_matrix.shape[0]
    scaled = sim_matrix / temperature
    
    # Numerical stability
    scaled = scaled - np.max(scaled, axis=1, keepdims=True)
    
    exp_sim = np.exp(scaled)
    
    # For each query i, positive is passage i (diagonal)
    # Loss = -log(exp(sim(qi, pi+) / tau) / sum_j(exp(sim(qi, pj) / tau)))
    log_probs = scaled[np.arange(B), np.arange(B)] - np.log(np.sum(exp_sim, axis=1))
    
    loss = -np.mean(log_probs)
    return loss

# --- 2d: Temperature effect ---
print("\n--- Temperature Effect on InfoNCE Loss ---")
temps = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
rows = []
for tau in temps:
    loss = infonce_loss(sim_matrix, temperature=tau)
    # Also compute accuracy: is diagonal the max in each row?
    scaled = sim_matrix / tau
    preds = np.argmax(scaled, axis=1)
    acc = np.mean(preds == np.arange(B))
    rows.append([f'{tau:.2f}', f'{loss:.4f}', f'{acc:.0%}'])
print_table(['Temperature', 'InfoNCE Loss', 'Top-1 Acc'], rows, col_width=16)

# --- 2e: Hard negative effect ---
print("\n--- Hard Negative Mining Effect ---")
# Create easy vs hard negatives
easy_negatives = np.random.randn(B, d)
easy_negatives = l2_normalise(easy_negatives)

# Hard negatives: close to positive but not matching
hard_negatives = positives + 0.3 * np.random.randn(B, d)
hard_negatives = l2_normalise(hard_negatives)
# Shift hard negatives: each query's hard negative is the NEXT positive
hard_negatives = np.roll(positives, 1, axis=0) + 0.05 * np.random.randn(B, d)
hard_negatives = l2_normalise(hard_negatives)

# Compare losses with easy vs hard negatives
all_easy = np.hstack([positives[:, np.newaxis, :], 
                       easy_negatives[:, np.newaxis, :]]).reshape(B, 2, d)
all_hard = np.hstack([positives[:, np.newaxis, :],
                       hard_negatives[:, np.newaxis, :]]).reshape(B, 2, d)

sim_easy = np.array([[queries[i] @ all_easy[i, j] for j in range(2)] for i in range(B)])
sim_hard = np.array([[queries[i] @ all_hard[i, j] for j in range(2)] for i in range(B)])

print(f"Avg similarity to positive:        {np.mean(sim_easy[:,0]):.4f}")
print(f"Avg similarity to easy negative:   {np.mean(sim_easy[:,1]):.4f}")
print(f"Avg similarity to hard negative:   {np.mean(sim_hard[:,1]):.4f}")
print(f"Loss with easy negatives (tau=0.1): {infonce_loss(sim_matrix, 0.1):.4f}")

# Full sim matrix with hard negatives injected
sim_hard_full = sim_matrix.copy()
for i in range(B):
    j = (i + 1) % B
    sim_hard_full[i, j] = queries[i] @ hard_negatives[i]
print(f"Loss with hard negatives (tau=0.1): {infonce_loss(sim_hard_full, 0.1):.4f}")
print("\nHarder negatives → higher loss → stronger gradient signal → better learning")

# --- 2f: Gradient analysis ---
print("\n--- Gradient of InfoNCE w.r.t. similarity ---")
tau = 0.1
scaled = sim_matrix / tau
exp_sim = np.exp(scaled - np.max(scaled, axis=1, keepdims=True))
probs = exp_sim / exp_sim.sum(axis=1, keepdims=True)

# Gradient for positive pair (i,i): (1/tau)(P(j=i|qi) - 1)
# Gradient for negative pair (i,j): (1/tau)(P(j|qi))
grad_pos = (1/tau) * (probs[np.arange(B), np.arange(B)] - 1)
grad_neg_avg = (1/tau) * np.mean(probs - np.eye(B) * probs, axis=1)
print(f"Avg gradient for positive pairs: {np.mean(grad_pos):.4f}")
print(f"Avg gradient for negative pairs: {np.mean(grad_neg_avg):.4f}")
print("Gradient pushes positives closer (negative grad) and negatives apart (positive grad)")

## §3 Similarity Functions Compared

Three core similarity functions for retrieval:
- **Dot product:** $\text{sim}(q,p) = q \cdot p$
- **Cosine similarity:** $\text{sim}(q,p) = \frac{q \cdot p}{\|q\|\|p\|}$
- **Negative L2:** $\text{sim}(q,p) = -\|q - p\|_2^2$

For L2-normalised vectors: cosine = dot product and $-\|q-p\|^2 = 2(\text{cos} - 1)$.

In [ ]:
# === §3: Similarity Functions Compared ===

# --- 3a: Define similarity functions ---
def dot_product(q, p):
    return np.dot(q, p)

def cosine_similarity(q, p):
    norm_q = np.linalg.norm(q)
    norm_p = np.linalg.norm(p)
    if norm_q < 1e-10 or norm_p < 1e-10:
        return 0.0
    return np.dot(q, p) / (norm_q * norm_p)

def neg_l2_squared(q, p):
    return -np.sum((q - p) ** 2)

# --- 3b: Test with normalised vs unnormalised vectors ---
np.random.seed(42)
d = 768
q_raw = np.random.randn(d) * 2.0  # unnormalised
p_raw = np.random.randn(d) * 0.5  # different magnitude

q_norm = q_raw / np.linalg.norm(q_raw)
p_norm = p_raw / np.linalg.norm(p_raw)

print("--- Unnormalised vectors ---")
print(f"  ||q|| = {np.linalg.norm(q_raw):.4f},  ||p|| = {np.linalg.norm(p_raw):.4f}")
print(f"  Dot product:     {dot_product(q_raw, p_raw):.4f}")
print(f"  Cosine:          {cosine_similarity(q_raw, p_raw):.4f}")
print(f"  Neg L2 squared:  {neg_l2_squared(q_raw, p_raw):.4f}")

print("\n--- L2-normalised vectors ---")
print(f"  ||q|| = {np.linalg.norm(q_norm):.4f},  ||p|| = {np.linalg.norm(p_norm):.4f}")
print(f"  Dot product:     {dot_product(q_norm, p_norm):.4f}")
print(f"  Cosine:          {cosine_similarity(q_norm, p_norm):.4f}")
print(f"  Neg L2 squared:  {neg_l2_squared(q_norm, p_norm):.4f}")
print(f"  2*(cos - 1):     {2*(cosine_similarity(q_norm, p_norm) - 1):.4f}")
print(f"  ✓ neg_l2² == 2*(cos-1) for normalised vectors")

# --- 3c: Ranking consistency ---
print("\n--- Ranking Consistency Across Metrics ---")
n_docs = 20
docs = np.random.randn(n_docs, d)
query = np.random.randn(d)

# L2-normalise everything
query_n = query / np.linalg.norm(query)
docs_n = docs / np.linalg.norm(docs, axis=1, keepdims=True)

ranks_dot = np.argsort([dot_product(query_n, docs_n[i]) for i in range(n_docs)])[::-1]
ranks_cos = np.argsort([cosine_similarity(query_n, docs_n[i]) for i in range(n_docs)])[::-1]
ranks_l2 = np.argsort([neg_l2_squared(query_n, docs_n[i]) for i in range(n_docs)])[::-1]

print(f"Top-5 by dot product: {list(ranks_dot[:5])}")
print(f"Top-5 by cosine:      {list(ranks_cos[:5])}")
print(f"Top-5 by neg L2²:     {list(ranks_l2[:5])}")
print(f"Rankings identical (normalised): {np.all(ranks_dot == ranks_cos) and np.all(ranks_cos == ranks_l2)}")

# --- 3d: Without normalisation, rankings can differ ---
ranks_dot_raw = np.argsort([dot_product(query, docs[i]) for i in range(n_docs)])[::-1]
ranks_cos_raw = np.argsort([cosine_similarity(query, docs[i]) for i in range(n_docs)])[::-1]
ranks_l2_raw = np.argsort([neg_l2_squared(query, docs[i]) for i in range(n_docs)])[::-1]

print(f"\n--- Without Normalisation ---")
print(f"Top-5 by dot product: {list(ranks_dot_raw[:5])}")
print(f"Top-5 by cosine:      {list(ranks_cos_raw[:5])}")
print(f"Top-5 by neg L2²:     {list(ranks_l2_raw[:5])}")
print(f"Rankings identical (raw): {np.all(ranks_dot_raw == ranks_cos_raw) and np.all(ranks_cos_raw == ranks_l2_raw)}")
print("\n⚠ Different metrics give different rankings for unnormalised vectors!")
print("  → Always L2-normalise, then all three are monotonically equivalent.")

# --- 3e: Speed comparison ---
import time

n_docs_speed = 100000
d_speed = 768
docs_big = np.random.randn(n_docs_speed, d_speed).astype(np.float32)
q_big = np.random.randn(d_speed).astype(np.float32)

# Dot product (matrix multiply)
t0 = time.perf_counter()
scores_dot = docs_big @ q_big
t_dot = time.perf_counter() - t0

# L2 distance
t0 = time.perf_counter()
scores_l2 = -np.sum((docs_big - q_big) ** 2, axis=1)
t_l2 = time.perf_counter() - t0

print(f"\n--- Speed: {n_docs_speed:,} docs × d={d_speed} ---")
print(f"  Dot product (matmul): {t_dot*1000:.1f} ms")
print(f"  L2 distance:          {t_l2*1000:.1f} ms")
print(f"  Dot product is ~{t_l2/t_dot:.1f}× faster (BLAS-optimised matmul)")

## §4 Inverted File Index (IVF) Simulation

Partition vector space into Voronoi cells via k-means clustering:
1. Cluster all document vectors into $n_{\text{list}}$ centroids
2. At query time, find $n_{\text{probe}}$ nearest centroids
3. Search only documents in those cells

Speedup ≈ $n_{\text{list}} / n_{\text{probe}}$; recall depends on how many cells probed.

In [ ]:
# === §4: Inverted File Index (IVF) Simulation ===

# --- 4a: Generate corpus vectors ---
np.random.seed(42)
n_docs = 10000
d = 64  # reduced for speed
corpus_vecs = np.random.randn(n_docs, d).astype(np.float64)
corpus_vecs /= np.linalg.norm(corpus_vecs, axis=1, keepdims=True)

# --- 4b: K-means clustering (simplified) ---
def simple_kmeans(data, k, n_iter=20):
    """Simple k-means for IVF cell assignment."""
    n = data.shape[0]
    # Random initialisation
    indices = np.random.choice(n, k, replace=False)
    centroids = data[indices].copy()
    
    for iteration in range(n_iter):
        # Assign each vector to nearest centroid
        # distances: [n, k]
        dists = np.sum((data[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
        assignments = np.argmin(dists, axis=1)
        
        # Update centroids
        new_centroids = np.zeros_like(centroids)
        for c in range(k):
            mask = assignments == c
            if np.any(mask):
                new_centroids[c] = data[mask].mean(axis=0)
            else:
                new_centroids[c] = centroids[c]
        
        shift = np.sum((new_centroids - centroids) ** 2)
        centroids = new_centroids
        if shift < 1e-8:
            break
    
    return centroids, assignments

n_list = 100  # number of Voronoi cells
centroids, cell_assignments = simple_kmeans(corpus_vecs, n_list, n_iter=15)
print(f"IVF index: {n_docs} vectors in {n_list} cells")

# Cell size distribution
cell_sizes = [np.sum(cell_assignments == c) for c in range(n_list)]
print(f"Cell sizes — min: {min(cell_sizes)}, max: {max(cell_sizes)}, "
      f"mean: {np.mean(cell_sizes):.1f}, std: {np.std(cell_sizes):.1f}")

# Build inverted lists
inverted_lists = {}
for c in range(n_list):
    inverted_lists[c] = np.where(cell_assignments == c)[0]

# --- 4c: IVF search ---
def ivf_search(query, centroids, inverted_lists, corpus_vecs, n_probe=10, top_k=10):
    """Search top-k nearest neighbours using IVF index."""
    # Find n_probe nearest centroids
    centroid_dists = np.sum((centroids - query) ** 2, axis=1)
    nearest_cells = np.argsort(centroid_dists)[:n_probe]
    
    # Collect candidate indices from probed cells
    candidates = []
    for c in nearest_cells:
        candidates.extend(inverted_lists[c])
    
    if len(candidates) == 0:
        return [], []
    
    candidates = np.array(candidates)
    # Score all candidates
    scores = corpus_vecs[candidates] @ query
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return candidates[top_indices], scores[top_indices]

# --- 4d: Brute force baseline ---
query = np.random.randn(d).astype(np.float64)
query /= np.linalg.norm(query)

exact_scores = corpus_vecs @ query
exact_top_k = np.argsort(exact_scores)[::-1][:10]
exact_top_ids = set(exact_top_k)

print(f"\nExact top-10: {list(exact_top_k)}")

# --- 4e: Recall vs n_probe ---
print("\n--- Recall@10 vs n_probe ---")
rows = []
for n_probe in [1, 2, 5, 10, 20, 50, 100]:
    ivf_ids, ivf_scores = ivf_search(query, centroids, inverted_lists, corpus_vecs, 
                                      n_probe=n_probe, top_k=10)
    recall = len(set(ivf_ids) & exact_top_ids) / len(exact_top_ids)
    n_candidates = sum(len(inverted_lists[c]) for c in np.argsort(
        np.sum((centroids - query)**2, axis=1))[:n_probe])
    speedup = n_docs / max(n_candidates, 1)
    rows.append([n_probe, f'{recall:.0%}', n_candidates, f'{speedup:.1f}x'])
print_table(['n_probe', 'Recall@10', 'Candidates', 'Speedup'], rows, col_width=14)
print("\nMore probes → higher recall but slower search")
print(f"Sweet spot: n_probe ≈ {int(np.sqrt(n_list))} for good recall-speed tradeoff")

## §5 Product Quantization (PQ) for Vector Compression

Compress $d$-dimensional vectors to $m$ bytes:
1. Split vector into $m$ subvectors of $d/m$ dimensions
2. Learn $K=256$ centroids per subspace via k-means
3. Represent each subvector by its centroid index (1 byte)

Storage: $m$ bytes per vector vs $4d$ bytes (FP32). Compression ratio: $4d/m$.

In [ ]:
# === §5: Product Quantization (PQ) for Vector Compression ===

np.random.seed(42)
n_vecs = 5000
d = 64
m = 8  # number of subspaces
K = 256  # centroids per subspace
sub_d = d // m  # dimensions per subspace

data = np.random.randn(n_vecs, d).astype(np.float64)

print(f"PQ configuration: d={d}, m={m} subspaces, K={K} centroids, sub_d={sub_d}")
print(f"Original storage: {d * 4} bytes/vec (FP32)")
print(f"PQ storage:       {m} bytes/vec")
print(f"Compression:      {d * 4 / m:.0f}×")

# --- 5a: Train PQ codebooks ---
def train_pq(data, m, K, sub_d, n_iter=10):
    """Train product quantization codebooks."""
    n = data.shape[0]
    codebooks = []  # m codebooks, each [K, sub_d]
    codes = np.zeros((n, m), dtype=np.uint8)
    
    for s in range(m):
        # Extract subvectors for this subspace
        start = s * sub_d
        end = start + sub_d
        sub_data = data[:, start:end]
        
        # K-means on subvectors
        # Simple k-means (using random init)
        idx = np.random.choice(n, K, replace=False)
        centroids = sub_data[idx].copy()
        
        for _ in range(n_iter):
            dists = np.sum((sub_data[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
            assignments = np.argmin(dists, axis=1)
            for c in range(K):
                mask = assignments == c
                if np.any(mask):
                    centroids[c] = sub_data[mask].mean(axis=0)
        
        # Final assignment
        dists = np.sum((sub_data[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
        codes[:, s] = np.argmin(dists, axis=1).astype(np.uint8)
        codebooks.append(centroids)
    
    return codebooks, codes

codebooks, codes = train_pq(data, m, K, sub_d, n_iter=8)
print(f"\nTrained {m} codebooks, each [{K}, {sub_d}]")

# --- 5b: Reconstruct and measure error ---
def pq_reconstruct(codes, codebooks, sub_d):
    """Reconstruct vectors from PQ codes."""
    n = codes.shape[0]
    m = codes.shape[1]
    d = m * sub_d
    reconstructed = np.zeros((n, d))
    for s in range(m):
        start = s * sub_d
        end = start + sub_d
        reconstructed[:, start:end] = codebooks[s][codes[:, s]]
    return reconstructed

reconstructed = pq_reconstruct(codes, codebooks, sub_d)
errors = np.sqrt(np.sum((data - reconstructed) ** 2, axis=1))
print(f"\nReconstruction error (L2):")
print(f"  Mean:   {np.mean(errors):.4f}")
print(f"  Median: {np.median(errors):.4f}")
print(f"  Max:    {np.max(errors):.4f}")

# Relative error
rel_errors = errors / np.linalg.norm(data, axis=1)
print(f"  Mean relative error: {np.mean(rel_errors):.4%}")

# --- 5c: Asymmetric distance computation ---
def pq_asymmetric_distance(query, codes, codebooks, sub_d):
    """Compute distances from query to PQ-encoded vectors using lookup table."""
    m = len(codebooks)
    K = codebooks[0].shape[0]
    n = codes.shape[0]
    
    # Precompute distance table: [m, K]
    dist_table = np.zeros((m, K))
    for s in range(m):
        start = s * sub_d
        end = start + sub_d
        q_sub = query[start:end]
        # Distance from query subvector to each centroid
        dist_table[s] = np.sum((codebooks[s] - q_sub) ** 2, axis=1)
    
    # Compute total distance using lookup
    distances = np.zeros(n)
    for s in range(m):
        distances += dist_table[s, codes[:, s]]
    
    return distances

query = np.random.randn(d)
pq_dists = pq_asymmetric_distance(query, codes, codebooks, sub_d)
exact_dists = np.sum((data - query) ** 2, axis=1)

# Compare rankings
pq_top10 = set(np.argsort(pq_dists)[:10])
exact_top10 = set(np.argsort(exact_dists)[:10])
recall = len(pq_top10 & exact_top10) / 10

print(f"\n--- Asymmetric Distance Search ---")
print(f"Recall@10 (PQ vs exact): {recall:.0%}")
print(f"Rank correlation (top 100): {np.corrcoef(np.argsort(pq_dists)[:100], np.argsort(exact_dists)[:100])[0,1]:.4f}")

# --- 5d: Storage comparison ---
print(f"\n--- Storage Summary ---")
headers = ['Config', 'Bytes/Vec', 'Total (5K)', 'Compression']
rows = [
    ['FP32', d*4, fmt_bytes(n_vecs*d*4), '1×'],
    ['FP16', d*2, fmt_bytes(n_vecs*d*2), '2×'],
    [f'PQ (m={m})', m, fmt_bytes(n_vecs*m), f'{d*4//m}×'],
    [f'PQ (m={m//2})', m//2, fmt_bytes(n_vecs*m//2), f'{d*4//(m//2):.0f}×'],
]
print_table(headers, rows, col_width=16)

## §6 HNSW — Hierarchical Navigable Small World Graph

Graph-based ANN index with logarithmic query complexity.
- Multiple layers: top = sparse long-range; bottom = dense all-nodes
- Greedy search descends layers; explores $\text{ef\_search}$ candidates at bottom
- Each node connected to $M$ nearest neighbours per layer

Parameters: $M$ (connections per node), $\text{ef\_construction}$, $\text{ef\_search}$.

In [ ]:
# === §6: HNSW Graph Search Simulation ===

np.random.seed(42)
n_nodes = 2000
d = 32

data = np.random.randn(n_nodes, d).astype(np.float64)
data /= np.linalg.norm(data, axis=1, keepdims=True)

# --- 6a: Build simplified HNSW ---
class SimpleHNSW:
    """Simplified HNSW for educational purposes."""
    
    def __init__(self, d, M=16, max_level=4):
        self.d = d
        self.M = M
        self.max_level = max_level
        self.nodes = []       # list of vectors
        self.graphs = [{}     # adjacency lists per level
                       for _ in range(max_level + 1)]
        self.entry_point = None
        self.node_levels = []
    
    def _random_level(self):
        """Assign level probabilistically (geometric distribution)."""
        level = 0
        while np.random.random() < 1.0 / self.M and level < self.max_level:
            level += 1
        return level
    
    def _distance(self, a, b):
        return -np.dot(a, b)  # negative dot product (lower = more similar)
    
    def add(self, vec):
        """Insert a vector into the HNSW graph."""
        node_id = len(self.nodes)
        self.nodes.append(vec)
        level = self._random_level()
        self.node_levels.append(level)
        
        # Initialise adjacency for this node at all its levels
        for l in range(level + 1):
            self.graphs[l][node_id] = set()
        
        if self.entry_point is None:
            self.entry_point = node_id
            return node_id
        
        # Greedily descend from top to node's level + 1
        current = self.entry_point
        for l in range(self.max_level, level + 1, -1):
            if current in self.graphs[l]:
                current = self._greedy_closest(vec, current, l)
        
        # At each of node's levels, find M nearest and connect
        for l in range(min(level, self.max_level), -1, -1):
            # Find candidates at this level
            candidates = self._search_layer(vec, current, self.M * 2, l)
            # Connect to M nearest
            neighbours = sorted(candidates, key=lambda x: self._distance(vec, self.nodes[x]))[:self.M]
            for nb in neighbours:
                self.graphs[l][node_id].add(nb)
                if nb in self.graphs[l]:
                    self.graphs[l][nb].add(node_id)
                    # Prune if over capacity
                    if len(self.graphs[l][nb]) > self.M:
                        # Keep M closest
                        nb_vec = self.nodes[nb]
                        sorted_nbs = sorted(self.graphs[l][nb], 
                                           key=lambda x: self._distance(nb_vec, self.nodes[x]))
                        self.graphs[l][nb] = set(sorted_nbs[:self.M])
            if candidates:
                current = candidates[0]
        
        # Update entry point if new node is at higher level
        if level > self.node_levels[self.entry_point]:
            self.entry_point = node_id
        
        return node_id
    
    def _greedy_closest(self, query, start, level):
        """Greedy walk to closest node at given level."""
        current = start
        while True:
            best = current
            best_dist = self._distance(query, self.nodes[current])
            for nb in self.graphs[level].get(current, set()):
                d = self._distance(query, self.nodes[nb])
                if d < best_dist:
                    best = nb
                    best_dist = d
            if best == current:
                break
            current = best
        return current
    
    def _search_layer(self, query, entry, ef, level):
        """Search a single layer; return ef nearest candidates."""
        visited = {entry}
        candidates = [(self._distance(query, self.nodes[entry]), entry)]
        results = list(candidates)
        
        import heapq
        heapq.heapify(candidates)
        
        while candidates:
            dist_c, c = heapq.heappop(candidates)
            # If closest candidate is farther than farthest result, stop
            if len(results) >= ef and dist_c > max(r[0] for r in results):
                break
            for nb in self.graphs[level].get(c, set()):
                if nb not in visited:
                    visited.add(nb)
                    d = self._distance(query, self.nodes[nb])
                    if len(results) < ef or d < max(r[0] for r in results):
                        heapq.heappush(candidates, (d, nb))
                        results.append((d, nb))
                        if len(results) > ef * 2:
                            results = sorted(results)[:ef]
        
        results = sorted(results)[:ef]
        return [r[1] for r in results]
    
    def search(self, query, k=10, ef_search=50):
        """Search for k nearest neighbours."""
        current = self.entry_point
        # Descend from top layer
        for l in range(self.max_level, 0, -1):
            if current in self.graphs[l]:
                current = self._greedy_closest(query, current, l)
        
        # Search bottom layer with ef_search candidates
        candidates = self._search_layer(query, current, ef_search, 0)
        # Return top-k
        candidates_scored = [(self._distance(query, self.nodes[c]), c) for c in candidates]
        candidates_scored.sort()
        return [(c, -d) for d, c in candidates_scored[:k]]

# Build index
hnsw = SimpleHNSW(d=d, M=16, max_level=4)
for i in range(n_nodes):
    hnsw.add(data[i])
print(f"HNSW index built: {n_nodes} nodes, M={hnsw.M}")

# Level distribution
level_counts = np.bincount(hnsw.node_levels, minlength=hnsw.max_level+1)
print(f"Level distribution: {dict(enumerate(level_counts))}")

# --- 6b: Search and compare with brute force ---
query = np.random.randn(d)
query /= np.linalg.norm(query)

# Exact search
exact_scores = data @ query
exact_top10 = set(np.argsort(exact_scores)[::-1][:10])

# HNSW search
results = hnsw.search(query, k=10, ef_search=50)
hnsw_top10 = set(r[0] for r in results)

recall = len(hnsw_top10 & exact_top10) / 10
print(f"\nRecall@10 (ef_search=50): {recall:.0%}")

# --- 6c: ef_search vs recall tradeoff ---
print("\n--- ef_search vs Recall@10 ---")
rows = []
for ef in [10, 20, 50, 100, 200, 500]:
    results = hnsw.search(query, k=10, ef_search=ef)
    hnsw_ids = set(r[0] for r in results)
    r = len(hnsw_ids & exact_top10) / 10
    rows.append([ef, f'{r:.0%}', f'~{ef}', f'{n_nodes/max(ef,1):.1f}x'])
print_table(['ef_search', 'Recall@10', 'Candidates', 'Speedup'], rows, col_width=14)

# --- 6d: Memory estimation ---
print("\n--- HNSW Memory Estimation ---")
for M in [8, 16, 32, 64]:
    # Each node stores M connections per level + vector
    # Avg levels ≈ 1 + 1/ln(M)
    avg_levels = 1 + 1 / np.log(M)
    mem_graph = n_nodes * avg_levels * M * 4  # 4 bytes per neighbor ID
    mem_vecs = n_nodes * d * 4  # FP32 vectors
    total = mem_graph + mem_vecs
    print(f"  M={M:2d}: graph={fmt_bytes(mem_graph)}, vectors={fmt_bytes(mem_vecs)}, "
          f"total={fmt_bytes(total)}")

## §7 HyDE — Hypothetical Document Embeddings

Query transformation technique (Gao et al. 2022):
1. Given query $q$, generate a **hypothetical answer** $h$ using LLM
2. Embed $h$ instead of $q$: $\hat{h} = E(h)$
3. Search with $\hat{h}$ — closer to document distribution than query distribution

The key insight: queries and documents live in different embedding distributions.
HyDE bridges this gap by transforming a query into the document distribution.

## §8 Reciprocal Rank Fusion (RRF)

Combine multiple ranking systems without score normalisation:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

$k=60$ is standard; prevents excessive weight on top ranks.
RRF works with any ranker; robust to score distribution differences.

In [ ]:
# === §8: Reciprocal Rank Fusion (RRF) ===

np.random.seed(42)

# --- 8a: Create two ranking systems ---
n_docs = 20
doc_ids = [f'D{i:02d}' for i in range(n_docs)]

# BM25 ranks (lexical)
bm25_order = np.random.permutation(n_docs)
# Dense retrieval ranks (semantic)
dense_order = np.random.permutation(n_docs)

# Make some overlap: both systems rank doc 5 and doc 12 highly
bm25_order = np.array([5, 3, 12, 7, 0, 15, 8, 2, 19, 11, 
                        1, 4, 6, 9, 10, 13, 14, 16, 17, 18])
dense_order = np.array([12, 5, 9, 0, 7, 3, 15, 1, 11, 8,
                         2, 19, 4, 6, 10, 13, 14, 16, 17, 18])

print("BM25 ranking:  ", [doc_ids[i] for i in bm25_order[:10]])
print("Dense ranking: ", [doc_ids[i] for i in dense_order[:10]])

# --- 8b: RRF computation ---
def rrf_fusion(rankings, k=60):
    """Reciprocal Rank Fusion across multiple ranking systems."""
    scores = {}
    for ranking in rankings:
        for rank_pos, doc_id in enumerate(ranking):
            if doc_id not in scores:
                scores[doc_id] = 0.0
            scores[doc_id] += 1.0 / (k + rank_pos + 1)  # rank is 1-indexed
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

rrf_results = rrf_fusion([bm25_order, dense_order], k=60)

print("\n--- RRF Fusion (k=60) ---")
headers = ['RRF Rank', 'Doc', 'RRF Score', 'BM25 Rank', 'Dense Rank']
rows = []
for rank, (doc_idx, score) in enumerate(rrf_results[:10]):
    bm25_rank = np.where(bm25_order == doc_idx)[0][0] + 1
    dense_rank = np.where(dense_order == doc_idx)[0][0] + 1
    rows.append([rank+1, doc_ids[doc_idx], f'{score:.6f}', bm25_rank, dense_rank])
print_table(headers, rows, col_width=12)

# --- 8c: Effect of k parameter ---
print("\n--- Effect of k on RRF ---")
print("Example: Doc ranked #1 by BM25, #10 by dense")
for k in [1, 10, 30, 60, 100, 500]:
    score_rank1 = 1.0 / (k + 1) + 1.0 / (k + 10)
    score_rank5 = 1.0 / (k + 5) + 1.0 / (k + 5)
    ratio = (1.0 / (k + 1)) / (1.0 / (k + 10))
    print(f"  k={k:3d}: score(rank 1+10) = {score_rank1:.6f}, "
          f"score(rank 5+5) = {score_rank5:.6f}, "
          f"top-rank boost = {ratio:.2f}×")
print("\nSmaller k → more weight on top-ranked docs; larger k → more uniform")

# --- 8d: RRF vs linear combination ---
print("\n--- RRF vs Linear Score Fusion ---")
bm25_scores = 1.0 / (1 + np.arange(n_docs))  # score = 1/rank
dense_scores = np.random.rand(n_docs) * 2  # arbitrary scale

# Normalise for linear combination
def min_max_normalise(scores):
    mn, mx = np.min(scores), np.max(scores)
    if mx - mn < 1e-10:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)

bm25_norm = min_max_normalise(bm25_scores[bm25_order])
dense_norm = min_max_normalise(dense_scores[dense_order])

# Map back to doc IDs
linear_scores = {}
for i, doc_idx in enumerate(bm25_order):
    linear_scores[doc_idx] = linear_scores.get(doc_idx, 0) + 0.5 * bm25_norm[i]
for i, doc_idx in enumerate(dense_order):
    linear_scores[doc_idx] = linear_scores.get(doc_idx, 0) + 0.5 * dense_norm[i]

linear_ranking = sorted(linear_scores.items(), key=lambda x: x[1], reverse=True)
rrf_ranking = [doc_idx for doc_idx, _ in rrf_results]
linear_ranking_ids = [doc_idx for doc_idx, _ in linear_ranking]

print(f"RRF top-5:    {[doc_ids[i] for i in rrf_ranking[:5]]}")
print(f"Linear top-5: {[doc_ids[i] for i in linear_ranking_ids[:5]]}")
overlap = len(set(rrf_ranking[:5]) & set(linear_ranking_ids[:5]))
print(f"Top-5 overlap: {overlap}/5")
print("\nRRF advantage: no score normalisation needed; robust to different scales")

# --- 8e: Three-way fusion ---
print("\n--- Three-Way Fusion (BM25 + Dense + ColBERT) ---")
colbert_order = np.array([9, 12, 5, 0, 3, 7, 15, 8, 2, 1,
                           11, 19, 4, 6, 10, 13, 14, 16, 17, 18])

rrf_3way = rrf_fusion([bm25_order, dense_order, colbert_order], k=60)
rrf_2way = rrf_fusion([bm25_order, dense_order], k=60)

print("2-way RRF top-5: ", [doc_ids[i] for i, _ in rrf_2way[:5]])
print("3-way RRF top-5: ", [doc_ids[i] for i, _ in rrf_3way[:5]])
print("\nAdding more retrieval systems improves consensus ranking")

## §9 NDCG, MRR, and Retrieval Metrics

Standard information retrieval evaluation metrics:

**MRR:** $\frac{1}{|Q|}\sum_q \frac{1}{\text{rank}_q}$

**DCG@k:** $\sum_{i=1}^{k} \frac{2^{\text{rel}_i}-1}{\log_2(i+1)}$

**NDCG@k:** $\frac{\text{DCG@k}}{\text{IDCG@k}}$ where IDCG is DCG of ideal ranking.

**MAP:** mean of average precision across queries.

In [ ]:
# === §9: NDCG, MRR, and Retrieval Metrics ===

# --- 9a: MRR computation ---
def mrr(ranked_results):
    """Mean Reciprocal Rank across multiple queries.
    ranked_results: list of lists, each containing 0/1 relevance per position.
    """
    rr_sum = 0.0
    for results in ranked_results:
        for i, rel in enumerate(results):
            if rel > 0:
                rr_sum += 1.0 / (i + 1)
                break
    return rr_sum / len(ranked_results)

# Example: 5 queries
query_results = [
    [0, 1, 0, 0, 0],  # first relevant at position 2 → RR = 1/2
    [1, 0, 0, 0, 0],  # first relevant at position 1 → RR = 1/1
    [0, 0, 1, 0, 0],  # first relevant at position 3 → RR = 1/3
    [0, 0, 0, 1, 0],  # first relevant at position 4 → RR = 1/4
    [1, 1, 0, 0, 0],  # first relevant at position 1 → RR = 1/1
]

mrr_score = mrr(query_results)
print(f"MRR = {mrr_score:.4f}")
print(f"  = (1/2 + 1/1 + 1/3 + 1/4 + 1/1) / 5 = {(0.5+1+1/3+0.25+1)/5:.4f}")

# --- 9b: DCG and NDCG ---
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at rank k."""
    relevances = np.asarray(relevances)[:k]
    gains = 2**relevances - 1
    discounts = np.log2(np.arange(1, len(relevances) + 1) + 1)
    return np.sum(gains / discounts)

def ndcg_at_k(relevances, k):
    """Normalised DCG at rank k."""
    dcg = dcg_at_k(relevances, k)
    ideal = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

# Worked example from notes
relevances = [3, 0, 2, 1, 0]
k = 5

print(f"\n--- NDCG@{k} Worked Example ---")
print(f"Relevance scores: {relevances}")

# Step-by-step DCG
print(f"\nDCG@{k} computation:")
gains = [2**r - 1 for r in relevances]
discounts = [np.log2(i + 2) for i in range(k)]
for i in range(k):
    print(f"  Position {i+1}: gain = 2^{relevances[i]}-1 = {gains[i]}, "
          f"discount = log₂({i+2}) = {discounts[i]:.4f}, "
          f"contribution = {gains[i]/discounts[i]:.4f}")
dcg = dcg_at_k(relevances, k)
print(f"  DCG@{k} = {dcg:.4f}")

# Ideal ranking
ideal = sorted(relevances, reverse=True)
idcg = dcg_at_k(ideal, k)
print(f"\nIdeal ranking: {ideal}")
print(f"  IDCG@{k} = {idcg:.4f}")
print(f"\nNDCG@{k} = DCG/IDCG = {dcg:.4f} / {idcg:.4f} = {dcg/idcg:.4f}")

# --- 9c: MAP computation ---
def average_precision(relevances):
    """Average precision for a single query."""
    n_relevant = sum(relevances)
    if n_relevant == 0:
        return 0.0
    precision_sum = 0.0
    relevant_seen = 0
    for i, rel in enumerate(relevances):
        if rel > 0:
            relevant_seen += 1
            precision_at_i = relevant_seen / (i + 1)
            precision_sum += precision_at_i
    return precision_sum / n_relevant

def mean_average_precision(all_results):
    return np.mean([average_precision(r) for r in all_results])

print(f"\n--- Mean Average Precision ---")
binary_results = [
    [1, 0, 1, 0, 1],  
    [0, 1, 0, 1, 0],
    [1, 1, 1, 0, 0],
]
for i, res in enumerate(binary_results):
    ap = average_precision(res)
    print(f"  Query {i+1}: results={res}, AP={ap:.4f}")
map_score = mean_average_precision(binary_results)
print(f"  MAP = {map_score:.4f}")

# --- 9d: Recall@k ---
def recall_at_k(retrieved, relevant, k):
    """Recall at position k."""
    retrieved_set = set(retrieved[:k])
    relevant_set = set(relevant)
    return len(retrieved_set & relevant_set) / len(relevant_set)

# Example
all_docs = list(range(100))
relevant = {3, 7, 15, 22, 45}
retrieved = [7, 3, 10, 22, 5, 45, 15, 8, 9, 1]

print(f"\n--- Recall@k ---")
for k in [1, 3, 5, 10]:
    r = recall_at_k(retrieved, relevant, k)
    print(f"  Recall@{k:2d} = {r:.2f}  (found {int(r*len(relevant))}/{len(relevant)})")

# --- 9e: Metric comparison across systems ---
print(f"\n--- Metric Comparison: BM25 vs Dense vs Hybrid ---")
# Simulated relevances for 3 systems
bm25_rels   = [3, 0, 2, 0, 1, 0, 0, 0, 1, 0]
dense_rels  = [0, 3, 0, 2, 0, 1, 1, 0, 0, 0]
hybrid_rels = [3, 2, 1, 0, 1, 0, 0, 1, 0, 0]

headers = ['Metric', 'BM25', 'Dense', 'Hybrid']
rows = [
    ['NDCG@5', f'{ndcg_at_k(bm25_rels, 5):.4f}', 
     f'{ndcg_at_k(dense_rels, 5):.4f}', f'{ndcg_at_k(hybrid_rels, 5):.4f}'],
    ['NDCG@10', f'{ndcg_at_k(bm25_rels, 10):.4f}', 
     f'{ndcg_at_k(dense_rels, 10):.4f}', f'{ndcg_at_k(hybrid_rels, 10):.4f}'],
    ['MRR', f'{mrr([[1 if r>0 else 0 for r in bm25_rels]]):.4f}',
     f'{mrr([[1 if r>0 else 0 for r in dense_rels]]):.4f}',
     f'{mrr([[1 if r>0 else 0 for r in hybrid_rels]]):.4f}'],
    ['AP', f'{average_precision([1 if r>0 else 0 for r in bm25_rels]):.4f}',
     f'{average_precision([1 if r>0 else 0 for r in dense_rels]):.4f}',
     f'{average_precision([1 if r>0 else 0 for r in hybrid_rels]):.4f}'],
]
print_table(headers, rows, col_width=12)
print("\nHybrid typically wins across metrics — combines best of both systems")

## §10 RAG Marginalisation and Probabilistic Retrieval

Full probabilistic RAG model marginalises over retrieved documents:

$$P(y \mid q) = \sum_{d \in \text{TopK}} P(d \mid q) \cdot P(y \mid q, d)$$

Convert retrieval scores to probabilities via softmax:
$$P(d_i \mid q) = \frac{\exp(s_i / \tau)}{\sum_j \exp(s_j / \tau)}$$

Information gain from retrieval:
$$\text{IG}(d; y \mid q) = H(y \mid q) - H(y \mid q, d)$$

In [ ]:
# === §10: RAG Marginalisation and Probabilistic Retrieval ===

# --- 10a: Score to probability conversion ---
retrieval_scores = np.array([2.1, 1.8, 0.9, 0.5, 0.2])
doc_names = ['D_A', 'D_B', 'D_C', 'D_D', 'D_E']
k = len(retrieval_scores)

print("--- Retrieval Score → Probability ---")
print(f"Raw scores: {fmt_vec(retrieval_scores)}")
for tau in [0.1, 0.5, 1.0, 2.0, 5.0]:
    probs = softmax(retrieval_scores, temperature=tau)
    print(f"  τ={tau:.1f}: [{', '.join(f'{p:.4f}' for p in probs)}]  "
          f"entropy={entropy(probs):.4f}")
print("Lower τ → more peaked (greedy); higher τ → more uniform (exploratory)")

# --- 10b: RAG marginalisation ---
print("\n--- RAG Marginalisation ---")
# Top-3 retrieved documents
top_k = 3
scores = np.array([2.1, 1.8, 0.9])
doc_labels = ['D_A', 'D_B', 'D_C']

# LLM log-probabilities for answer y given each document
llm_log_probs = np.array([-1.2, -2.5, -4.1])  # log P(y | q, d_i)
llm_probs = np.exp(llm_log_probs)

# P(d | q) via softmax
tau = 1.0
retrieval_probs = softmax(scores, temperature=tau)

print(f"Retrieval scores:  {fmt_vec(scores)}")
print(f"P(d|q) [τ={tau}]: [{', '.join(f'{p:.4f}' for p in retrieval_probs)}]")
print(f"LLM log P(y|q,d): {fmt_vec(llm_log_probs)}")
print(f"LLM P(y|q,d):     [{', '.join(f'{p:.4f}' for p in llm_probs)}]")

# Marginalise: P(y|q) = Σ P(d|q) * P(y|q,d)
marginal_prob = np.sum(retrieval_probs * llm_probs)
marginal_log_prob = np.log(marginal_prob)

print(f"\nMarginalised P(y|q) = Σ P(d|q) × P(y|q,d)")
for i in range(top_k):
    contrib = retrieval_probs[i] * llm_probs[i]
    print(f"  {doc_labels[i]}: {retrieval_probs[i]:.4f} × {llm_probs[i]:.4f} = {contrib:.6f}")
print(f"  Sum = {marginal_prob:.6f}")
print(f"  log P(y|q) = {marginal_log_prob:.4f}")

# --- 10c: Top-1 vs marginalised comparison ---
print("\n--- Top-1 vs Marginalised ---")
top1_prob = llm_probs[0]  # use only highest-scoring document
print(f"Top-1 P(y|q) = P(y|q,D_A) = {top1_prob:.6f} (log: {np.log(top1_prob):.4f})")
print(f"Marginalised P(y|q)       = {marginal_prob:.6f} (log: {marginal_log_prob:.4f})")
print(f"Improvement: {(marginal_prob/top1_prob - 1)*100:.2f}%")
print("Marginalisation always ≥ top-1 (Jensen's inequality)")

# --- 10d: Temperature sensitivity of marginalisation ---
print("\n--- Marginalisation vs Temperature ---")
rows = []
for tau in [0.1, 0.5, 1.0, 2.0, 5.0]:
    r_probs = softmax(scores, temperature=tau)
    marg_p = np.sum(r_probs * llm_probs)
    # Effective: which doc dominates?
    dominant = np.argmax(r_probs * llm_probs)
    rows.append([f'{tau:.1f}', f'{marg_p:.6f}', f'{np.log(marg_p):.4f}',
                 f'{entropy(r_probs):.4f}', doc_labels[dominant]])
print_table(['τ', 'P(y|q)', 'log P(y|q)', 'H(d|q)', 'Dominant'], rows, col_width=12)

# --- 10e: Information gain from retrieval ---
print("\n--- Information Gain from Retrieval ---")
# Model uncertainty without retrieval
vocab_size = 50000
# Base model: high entropy (uncertain)
base_logits = np.random.randn(vocab_size) * 0.5
base_probs = softmax(base_logits, temperature=1.0)
H_base = entropy(base_probs)

# After retrieval: model becomes more confident
boosted_logits = base_logits.copy()
# Boost a few tokens (the answer tokens)
answer_tokens = [42, 100, 555]
for t in answer_tokens:
    boosted_logits[t] += 5.0
retrieved_probs = softmax(boosted_logits, temperature=1.0)
H_retrieved = entropy(retrieved_probs)

ig = H_base - H_retrieved
print(f"H(y|q) without retrieval: {H_base:.4f} nats")
print(f"H(y|q,d) with retrieval:  {H_retrieved:.4f} nats")
print(f"Information gain IG(d;y|q) = {ig:.4f} nats")
print(f"  = {ig / np.log(2):.4f} bits")
print(f"Entropy reduction: {(1 - H_retrieved/H_base)*100:.1f}%")
print("Good retrieval drastically reduces answer uncertainty")

# --- 10f: Embedding space isotropy check ---
print("\n--- Embedding Space Isotropy ---")
d_emb = 768
n_vecs = 1000

# Isotropic: random unit vectors
isotropic = np.random.randn(n_vecs, d_emb)
isotropic /= np.linalg.norm(isotropic, axis=1, keepdims=True)

# Anisotropic: add a strong shared component (simulates BERT degeneration)
shared_component = np.random.randn(d_emb) * 3.0
anisotropic = np.random.randn(n_vecs, d_emb) * 0.3 + shared_component
anisotropic /= np.linalg.norm(anisotropic, axis=1, keepdims=True)

# Measure: average pairwise cosine similarity (should be ~0 for isotropic)
n_pairs = 500
pairs = np.random.choice(n_vecs, (n_pairs, 2), replace=True)
iso_cos = np.mean([isotropic[i] @ isotropic[j] for i, j in pairs if i != j])
aniso_cos = np.mean([anisotropic[i] @ anisotropic[j] for i, j in pairs if i != j])

print(f"Isotropic embeddings:  avg pairwise cosine = {iso_cos:.4f} (ideal ≈ 0)")
print(f"Anisotropic embeddings: avg pairwise cosine = {aniso_cos:.4f} (problematic!)")
print(f"\nAnisotropic space inflates similarity → retrieval quality degrades")
print("Fix: whitening (center + PCA rotation), or train with isotropy regularisation")

print()
print('=' * 65)
print('   THEORY NOTEBOOK COMPLETE')
print('=' * 65)
print()
print('Key Takeaways:')
print('  1. BM25: saturating TF + length normalisation; strong baseline')
print('  2. InfoNCE: temperature τ controls discrimination sharpness')
print('  3. L2-normalised → dot product = cosine = monotonic neg-L2')
print('  4. IVF: √N cells, probe ~1%; balances recall vs speedup')
print('  5. PQ: m bytes/vector; asymmetric distance via lookup table')
print('  6. HNSW: log(N) query; M and ef_search control recall')
print('  7. HyDE bridges query-document distribution gap')
print('  8. RRF: rank-based fusion; no normalisation; robust')
print('  9. NDCG: position-weighted gain; standard retrieval metric')
print(' 10. Marginalisation ≥ top-1; uses multiple retrieved docs')